<h2>Formating of DB-Data<h2>

In [1]:
from hilfsfunktionen.formating import get_unchanged_table, get_formated_table, get_table_with_keys, create_n_sentences
df_collection = {}
df_collection["unchanged_table"] = get_unchanged_table()
df_collection["formated_table"] = get_formated_table()
df_collection["formated_table_no_churn"] = df_collection["formated_table"].drop('churn', axis=1)
df_collection["formated_table_with_keys"] = get_table_with_keys(df_collection["formated_table"])
df_collection["formated_table_with_keys_no_churn"] = get_table_with_keys(df_collection["formated_table_no_churn"])
df_collection["n_sentences"] = {}
for n in [1, 5, 20, 50, 100]:
    df_collection["n_sentences"][f"{n}"] = create_n_sentences(df_collection["formated_table_with_keys_no_churn"], n)

#df_collection["formated_table_with_keys_no_churn"].head

<h2>Model Creation<h2>

In [2]:
from hilfsfunktionen.model_training import (
    ModelConfig, 
    ModelTrainer, 
    ModelRepository, 
    ExperimentRunner
)
from pathlib import Path

models = {}  

base_config = ModelConfig(
    vector_size = 50,      # Dimension der Wortvektoren; guter Kompromiss zwischen Ausdrucksstärke und Rechenaufwand
    window = 4,             # Kleines Kontextfenster; betont lokale, eher syntaktische Beziehungen
    epochs = 30,            # Viele Trainingsdurchläufe; bessere Anpassung, aber erhöhtes Overfitting-Risiko
    min_count = 1,          # Alle Wörter werden berücksichtigt; seltene Wörter sind jedoch oft verrauscht
    negative = 5,           # Anzahl negativer Beispiele; Standardwert mit gutem Qualitäts-/Zeit-Verhältnis
    sample = 0,             # Subsampling häufiger Wörter; verbessert Semantik, kann bei kleinen Korpora schaden
    hs = 0,                 # Hierarchical Softmax deaktiviert; Training erfolgt ausschließlich via Negative Sampling
    alpha = 0.025,          # Start-Lernrate; stabiler Trainingsbeginn, bei vielen Epochen kritisch
    seed = 1,               # Faester Zufallsstart; sorgt für Reproduzierbarkeit   
)

GRID = {
    "vector_size": [150],
    "window": [4],
    "sg": [0, 1],
    "epochs": [30],
    "negative": [5],
    "sample": [1e-3],
    "hs": [0],
    "alpha": [0.025],
    "seed": [1]
}

trainer = ModelTrainer()
repository = ModelRepository(base_dir=Path.cwd())
runner = ExperimentRunner(trainer, repository, base_config)


models = runner.run_grid_search(
    sentences=df_collection["n_sentences"]["20"],
    param_grid=GRID,
    #n_sentences_tokenized=df_collection["n_sentences"],
    num_sentences=10000, # Begrenzung der Trainingssätze; beschleunigt Training, limitiert Informationsgehalt
    verbose=False
)
    

<h2>Durchschnittliche Satzvektoren berechnen<h2>

In [3]:
from hilfsfunktionen.sentence_vectors import create_average_sentence_vectors

average_sentence_vectors = []

        
for model in models:
    vectors = create_average_sentence_vectors(
        model, 
        df_collection["formated_table_with_keys"], 
        base_dir=Path.cwd(), 
        verbose=False
    )
    average_sentence_vectors.append([vectors, model[1]])
#print(average_sentence_vectors[1])

<h2>Vektoren-Test Textbasiert<h2>

In [4]:
from hilfsfunktionen.weighting_cosine_similarities import (
    run_experiment2_weighting_analysis,
    select_top_churn_candidates,
    profile_top_churn_candidates
)

# 1. Gewichtungsvektoren definieren
weight_vectors = {
    'average': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'weighted_base': [0, 1, 3, 3, 1, 1, 1, 3, 3, 3, 1],  # Wie Vorlesung
    'weighted_demographic': [0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0],
    'weighted_financial': [0, 5, 0, 0, 0, 0, 5, 0, 0, 0, 5],
    'weighted_behavioral': [0, 0, 0, 0, 0, 5, 0, 5, 5, 5, 0]
}

# 2. Experiment ausführen (berechnet Churn-Scores)
results = run_experiment2_weighting_analysis(
    model_list=models,  # Ihr trainiertes Word2Vec-Modell
    df_with_keys=df_collection["formated_table_with_keys"],         # Ihre Kundendaten
    weight_vectors=weight_vectors,
    run_churn_propensity=True,         # ✅ WICHTIG: aktiviert Churn-Analyse
    churn_col='churn',                 # Name der Churn-Spalte
    churn_include_churned=False,       # Nur aktive Kunden bewerten
    churn_use_zscores=True,            # Z-Scores für Vergleichbarkeit
    verbose=False
)


ImportError: cannot import name 'weighted_proximity_avg' from 'hilfsfunktionen.cosine_funktions' (/app/pakete/hilfsfunktionen/src/hilfsfunktionen/cosine_funktions.py)

In [ ]:
for i in [0, 1]:
    # 3. Top-Kandidaten extrahieren (robuste Auswahl)
    churn_scores_df = results[f'exp2_phase3b_churn_scores_model{i}']
    top_candidates = results[f'exp2_phase3b_top10_churn_candidates_model{i}']
    
    print("Top 10 Churn-Risiko Kandidaten:")
    print(top_candidates)
    
    # 4. Detaillierte Profilerstellung
    profile = profile_top_churn_candidates(
        df_customers=df_collection["formated_table_with_keys"],
        candidate_ids=top10_fixed_current,
        id_col="key1",        # anpassen
        churn_col="churn",
        top_numeric_features=10
    )
    
    # Ausgabe:
    print("\n📊 Gesamt-Statistiken:")
    print(profile['overall_statistics'])
    
    print("\n🔥 Top Risikomerkmale global:")
    print(profile['global_risk_features'])
    
    print("\n👤 Kandidaten-Übersicht:")
    print(profile['candidate_overview'])
    
    print("\n📈 Numerische Feature-Treiber:")
    print(profile['numeric_top_drivers'])
    
    print("\n🏷️ Kategorische Risikofeatures:")
    print(profile['categorical_feature_profile'])

In [ ]:
import numpy as np

# Wir gehen durch beide Profile (Model 0 und Model 1)
for model_idx, result in enumerate(profiles):
    print(f"\n{'='*60}")
    print(f"ANALYSE FÜR MODEL {model_idx}")
    print(f"{'='*60}")
    
    # 1. Globale Churn-Rate
    baseline_rate = result["overall_statistics"]["overall_churn_rate"]
    print(f"\nBaseline Churn-Rate: {baseline_rate:.2%}")
    
    # 2. Durchschnittliche conditional_churn_rate der Kandidatenmerkmale
    df_cat = result["categorical_feature_profile"]
    
    # Für jeden Kandidaten: Durchschnittliche Churn-Rate seiner Merkmale
    candidate_risk_scores = {}
    for candidate_id in df_cat["key1"].unique():
        candidate_data = df_cat[df_cat["key1"] == candidate_id]
        avg_risk = candidate_data["conditional_churn_rate"].mean()
        candidate_risk_scores[candidate_id] = avg_risk
        print(f"Kandidat {candidate_id}: Durchschn. Merkmals-Churn-Rate = {avg_risk:.2%}")
    
    # Gesamtdurchschnitt aller Kandidaten
    overall_candidate_risk = np.mean(list(candidate_risk_scores.values()))
    print(f"\nDurchschn. Risiko aller Kandidaten (basierend auf Merkmalen): {overall_candidate_risk:.2%}")
    print(f"Verhältnis zur Baseline: {overall_candidate_risk/baseline_rate:.2f}x")
    
    # 3. Top-Risikomerkmale
    high_risk_features = df_cat[df_cat["churn_rate_ratio_vs_baseline"] > 1.5]
    print(f"\nAnzahl high-risk Merkmale (ratio > 1.5): {len(high_risk_features)}")
    
    # Gruppieren nach Feature und Wert
    if not high_risk_features.empty:
        feature_risk_summary = high_risk_features.groupby(["feature", "candidate_value"]).agg({
            "conditional_churn_rate": "mean",
            "churn_rate_ratio_vs_baseline": "mean",
            "churn_rate_delta_vs_baseline": "mean"
        }).sort_values("conditional_churn_rate", ascending=False)
        
        print("\nTop-Risikomerkmale:")
        print(feature_risk_summary.head(10))
    else:
        print("Keine high-risk Merkmale gefunden (ratio > 1.5).")
    
    # Optional: Analyse der gefilterten Daten
    print(f"\n{'='*40}")
    print(f"ANALYSE DER GEFILTERTEN DATEN (ohne 'key' Features)")
    print(f"{'='*40}")
    
    df_filtered_current = df_filtered[model_idx]
    print(f"Anzahl verbleibender Features nach Filterung: {len(df_filtered_current)}")
    
    if len(df_filtered_current) > 0:
        # Top 5 Features nach Churn-Rate Ratio
        top_features = df_filtered_current.sort_values("churn_rate_ratio_vs_baseline", ascending=False).head(5)
        print("\nTop 5 Risiko-Features (nach Ratio):")
        for idx, row in top_features.iterrows():
            print(f"  {row['feature']} = {row['candidate_value']}: Ratio = {row['churn_rate_ratio_vs_baseline']:.2f}, Churn-Rate = {row['conditional_churn_rate']:.2%}")

print(f"\n{'='*60}")
print("ANALYSE ABGESCHLOSSEN FÜR BEIDE MODELLE")
print(f"{'='*60}")

In [ ]:
from hilfsfunktionen.weighting_cosine_similarities import run_experiment2_weighting_analysis, compute_exp2_separation_matrix, compute_exp2_overlap_curves, compute_exp2_weight_impact_summary


# 1. Gewichtungsvektoren definieren (11 Features für Churn Dataset)
weight_vectors = {
    'average': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'weighted_base': [0, 1, 3, 3, 1, 1, 1, 3, 3, 3, 1],  # Wie Vorlesung
    'weighted_demographic': [0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0],
    'weighted_financial': [0, 5, 0, 0, 0, 0, 5, 0, 0, 0, 5],
    'weighted_behavioral': [0, 0, 0, 0, 0, 5, 0, 5, 5, 5, 0]
}



# 2. Experiment durchführen
results = run_experiment2_weighting_analysis(
    model_list=models,
    df_with_keys=df_collection["formated_table_with_keys"],
    weight_vectors=weight_vectors,
    metrics_to_use=['proximity_avg', 'proximity_topn_avg'],
    topn_n=3,
    subset_k=3,
    top_n=200,
    base_dir=".",
    churn_col="churn",
    run_churn_propensity=True,
    churn_include_churned=False,
    churn_use_zscores=True,
    verbose=True
)



In [ ]:
churn_scores = []
top10 = []
separation_matrix = []
overlap_curves = []
impact_summary = [] 

for i in [0, 1]:
    # 1. Churn Scores und Top 10 extrahieren
    churn_scores.append(results[f"exp2_phase3b_churn_scores_model{i}"])
    top10.append(results[f"exp2_phase3b_top10_churn_candidates_model{i}"])
    
    # 2. Darstellung 1 erstellen: Separation Matrix
    separation_matrix_result = compute_exp2_separation_matrix(
        phase4_results=results[f'exp2_phase4_separation_model{i}'],
        weight_names=list(weight_vectors.keys()),
        base_metrics=['avg', 'topn_avg']
    )
    separation_matrix.append(separation_matrix_result)
    
    print(f"\nSeparation Power Matrix für Model {i}:")
    print(separation_matrix_result)
    
    # 3. Darstellung 2 erstellen: Overlap Curves
    overlap_curves_result = compute_exp2_overlap_curves(
        rankings_dict=results[f'exp2_phase2_rankings_model{i}'],
        weight_names=list(weight_vectors.keys()),
        baseline_weight='average',
        metric_to_analyze='S_avg_average',  # Wähle beste Metrik aus Darstellung 1
        max_k=200,
        verbose=True
    )
    overlap_curves.append(overlap_curves_result)
    
    # 4. Impact-Summary berechnen
    impact_summary_result = compute_exp2_weight_impact_summary(
        overlap_curves_result=overlap_curves_result,
        threshold_k=50
    )
    impact_summary.append(impact_summary_result)
    
    print(f"\nImpact Summary für Model {i}:")
    print(impact_summary_result)

In [ ]:
from hilfsfunktionen.weighting_cosine_similarities import profile_top_churn_candidates

print(results.keys())

# Listen initialisieren
top10_df = []
top10_fixed = []
profiles = []
df_filtered = []

for i in [0, 1]:
    # 1. Top 10 DataFrame extrahieren
    top10_df_current = results[f"exp2_phase3b_top10_churn_candidates_model{i}"]
    top10_df.append(top10_df_current)
    
    # 2. IDs extrahieren und fixieren
    top10_ids = top10_df_current["Kunde_ID"].tolist()  # ggf. Customer_ID
    print(f"Top 10 IDs für Model {i}: {top10_ids}")
    
    top10_fixed_current = []
    for candidate in top10_ids:  # Hier top10_ids statt top10_ids[i]
        id_value = candidate + 1
        top10_fixed_current.append(f"key_"+str(id_value))
    
    top10_fixed.append(top10_fixed_current)
    print(f"Fixed IDs für Model {i}: {top10_fixed_current}")
    
    # 3. Profile erstellen
    profile_result = profile_top_churn_candidates(
        df_customers=df_collection["formated_table_with_keys"],
        candidate_ids=top10_fixed_current,
        id_col="key1",        # anpassen
        churn_col="churn",
        top_numeric_features=10
    )
    profiles.append(profile_result)
    
    # 4. DataFrame filtern (ohne 'key' in feature-Namen)
    df_filtered_current = profile_result["categorical_feature_profile"][
        ~profile_result["categorical_feature_profile"]['feature'].astype(str).str.contains('key', case=False, na=False)
    ]
    df_filtered.append(df_filtered_current)
    
    print(f"\nGefilterte kategorische Features für Model {i} (erste 5 Zeilen):")
print(df_filtered[0].head(50))

In [ ]:
from hilfsfunktionen.weighting_cosine_similarities import profile_top_churn_candidates

print(results.keys())

# Listen initialisieren
top10_df = []
top10_fixed = []
profiles = []
df_filtered = []

for i in [0, 1]:
    # 1. Top 10 DataFrame extrahieren
    top10_df_current = results[f"exp2_phase3b_top10_churn_candidates_model{i}"]
    top10_df.append(top10_df_current)
    
    # 2. IDs extrahieren und fixieren
    top10_ids = top10_df_current["Kunde_ID"].tolist()  # ggf. Customer_ID
    print(f"Top 10 IDs für Model {i}: {top10_ids}")
    
    top10_fixed_current = []
    for candidate in top10_ids:  # Hier top10_ids statt top10_ids[i]
        id_value = candidate + 1
        top10_fixed_current.append(f"key_"+str(id_value))
    
    top10_fixed.append(top10_fixed_current)
    print(f"Fixed IDs für Model {i}: {top10_fixed_current}")
    
    # 3. Profile erstellen
    profile_result = profile_top_churn_candidates(
        df_customers=df_collection["formated_table_with_keys"],
        candidate_ids=top10_fixed_current,
        id_col="key1",        # anpassen
        churn_col="churn",
        top_numeric_features=10
    )
    profiles.append(profile_result)
    
    # 4. DataFrame filtern (ohne 'key' in feature-Namen)
    df_filtered_current = profile_result["categorical_feature_profile"][
        ~profile_result["categorical_feature_profile"]['feature'].astype(str).str.contains('key', case=False, na=False)
    ]
    df_filtered.append(df_filtered_current)
    
    print(f"\nGefilterte kategorische Features für Model {i} (erste 5 Zeilen):")
    print(df_filtered[i].head(5))

In [ ]:
import numpy as np

# Wir gehen durch beide Profile (Model 0 und Model 1)
for model_idx, result in enumerate(profiles):
    print(f"\n{'='*60}")
    print(f"ANALYSE FÜR MODEL {model_idx}")
    print(f"{'='*60}")
    
    # 1. Globale Churn-Rate
    baseline_rate = result["overall_statistics"]["overall_churn_rate"]
    print(f"\nBaseline Churn-Rate: {baseline_rate:.2%}")
    
    # 2. Durchschnittliche conditional_churn_rate der Kandidatenmerkmale
    df_cat = result["categorical_feature_profile"]
    
    # Für jeden Kandidaten: Durchschnittliche Churn-Rate seiner Merkmale
    candidate_risk_scores = {}
    for candidate_id in df_cat["key1"].unique():
        candidate_data = df_cat[df_cat["key1"] == candidate_id]
        avg_risk = candidate_data["conditional_churn_rate"].mean()
        candidate_risk_scores[candidate_id] = avg_risk
        print(f"Kandidat {candidate_id}: Durchschn. Merkmals-Churn-Rate = {avg_risk:.2%}")
    
    # Gesamtdurchschnitt aller Kandidaten
    overall_candidate_risk = np.mean(list(candidate_risk_scores.values()))
    print(f"\nDurchschn. Risiko aller Kandidaten (basierend auf Merkmalen): {overall_candidate_risk:.2%}")
    print(f"Verhältnis zur Baseline: {overall_candidate_risk/baseline_rate:.2f}x")
    
    # 3. Top-Risikomerkmale
    high_risk_features = df_cat[df_cat["churn_rate_ratio_vs_baseline"] > 1.5]
    print(f"\nAnzahl high-risk Merkmale (ratio > 1.5): {len(high_risk_features)}")
    
    # Gruppieren nach Feature und Wert
    if not high_risk_features.empty:
        feature_risk_summary = high_risk_features.groupby(["feature", "candidate_value"]).agg({
            "conditional_churn_rate": "mean",
            "churn_rate_ratio_vs_baseline": "mean",
            "churn_rate_delta_vs_baseline": "mean"
        }).sort_values("conditional_churn_rate", ascending=False)
        
        print("\nTop-Risikomerkmale:")
        print(feature_risk_summary.head(10))
    else:
        print("Keine high-risk Merkmale gefunden (ratio > 1.5).")
    
    # Optional: Analyse der gefilterten Daten
    print(f"\n{'='*40}")
    print(f"ANALYSE DER GEFILTERTEN DATEN (ohne 'key' Features)")
    print(f"{'='*40}")
    
    df_filtered_current = df_filtered[model_idx]
    print(f"Anzahl verbleibender Features nach Filterung: {len(df_filtered_current)}")
    
    if len(df_filtered_current) > 0:
        # Top 5 Features nach Churn-Rate Ratio
        top_features = df_filtered_current.sort_values("churn_rate_ratio_vs_baseline", ascending=False).head(5)
        print("\nTop 5 Risiko-Features (nach Ratio):")
        for idx, row in top_features.iterrows():
            print(f"  {row['feature']} = {row['candidate_value']}: Ratio = {row['churn_rate_ratio_vs_baseline']:.2f}, Churn-Rate = {row['conditional_churn_rate']:.2%}")

print(f"\n{'='*60}")
print("ANALYSE ABGESCHLOSSEN FÜR BEIDE MODELLE")
print(f"{'='*60}")

<h2>Visualisierung<h2>

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)

def render_figure(fig, name, dpi=300, show=True):
    """
    Centralized figure handling:
    - saves in publication-quality formats
    - optionally displays in Jupyter
    - closes figure to avoid memory leaks
    """
    fig.savefig(FIGURE_DIR / f"{name}.pdf")
    fig.savefig(FIGURE_DIR / f"{name}.png", dpi=dpi)

    if show:
        plt.show()

    plt.close(fig)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle

def visualize_exp2_summary_enhanced(
    separation_matrix: pd.DataFrame,
    overlap_curves_result: dict,
    stats_df: pd.DataFrame,
    weight_vectors: dict,
    impact_summary: pd.DataFrame = None,  # NEU: Impact-Scores
    figsize=(18, 6)
):
    """
    Erweiterte kompakte Zusammenfassung mit mehr Informationsdichte.
    """
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(2, 4, height_ratios=[1, 0.15], hspace=0.35, wspace=0.4)
    
    # --- PLOT 1 (links oben): Separation Heatmap mit Ranking ---
    ax1 = fig.add_subplot(gs[0, 0])
    
    heatmap_data = separation_matrix.set_index('Gewichtung').copy()
    
    # Durchschnitt behalten für Ranking-Annotation
    has_avg = 'Durchschnitt' in heatmap_data.columns
    if has_avg:
        avg_col = heatmap_data['Durchschnitt'].copy()
        heatmap_data_plot = heatmap_data.drop(columns=['Durchschnitt'])
    else:
        heatmap_data_plot = heatmap_data
        avg_col = heatmap_data.mean(axis=1)
    
    # Heatmap
    sns.heatmap(
        heatmap_data_plot, 
        annot=True, 
        fmt='.3f',
        cmap='RdYlGn', 
        center=heatmap_data_plot.values.mean(),
        cbar_kws={'label': 'Separation Score', 'shrink': 0.8},
        linewidths=0.5, 
        ax=ax1, 
        vmin=0,
        cbar=True
    )
    
    # Ranking-Annotation (Ø-Score als Badge)
    for i, (idx, row) in enumerate(heatmap_data_plot.iterrows()):
        avg_score = avg_col.iloc[i]
        rank = avg_col.rank(ascending=False).iloc[i]
        
        # Badge rechts neben Heatmap
        ax1.text(
            len(heatmap_data_plot.columns) + 0.5, 
            i + 0.5, 
            f'#{int(rank)}\n{avg_score:.3f}',
            ha='center', 
            va='center',
            fontsize=8,
            fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='gold', alpha=0.3)
        )
    
    ax1.set_title('Separation Power Matrix\n(mit Ranking nach Ø-Score)', 
                  fontsize=11, fontweight='bold', pad=10)
    ax1.set_xlabel('Metrik', fontsize=9)
    ax1.set_ylabel('Gewichtung', fontsize=9)
    
    # --- PLOT 2 (Mitte oben): Overlap-Kurven mit Flächenfüllung ---
    ax2 = fig.add_subplot(gs[0, 1:3])
    
    curves = overlap_curves_result['curves']
    optimal = overlap_curves_result['optimal_curve']
    x = np.array(range(1, len(optimal) + 1))
    
    # Optimal-Kurve
    ax2.plot(x, optimal, 'k--', linewidth=2.5, label='Optimal', alpha=0.8, zorder=10)
    
    # Farben und Sortierung nach Impact
    colors = {
        'weighted_base': '#E74C3C',
        'weighted_demographic': '#3498DB',
        'weighted_financial': '#2ECC71',
        'weighted_behavioral': '#F39C12',
        'weighted_custom': '#9B59B6'
    }
    
    # Sortiere nach durchschnittlicher Abweichung (stärkster Impact zuerst)
    curve_impacts = {}
    for weight_name, curve in curves.items():
        avg_dev = np.mean([optimal[i] - curve[i] for i in range(len(curve))])
        curve_impacts[weight_name] = avg_dev
    
    sorted_curves = sorted(curve_impacts.items(), key=lambda x: x[1], reverse=True)
    
    # Plot Kurven mit Fläche zur Optimal-Kurve
    for weight_name, _ in sorted_curves:
        curve = curves[weight_name]
        color = colors.get(weight_name, '#999999')
        
        # Kurve
        line = ax2.plot(x, curve, linewidth=2, label=weight_name, 
                       color=color, alpha=0.85, zorder=5)
        
        # Fläche zwischen Kurve und Optimal (zeigt Impact visuell)
        ax2.fill_between(x, curve, optimal, 
                        color=color, alpha=0.1, zorder=1)
    
    # Vertikale Linien bei wichtigen K-Werten
    for k in [10, 50, 100]:
        ax2.axvline(x=k, color='gray', linewidth=0.8, linestyle=':', alpha=0.4)
        ax2.text(k, max(optimal) * 0.95, f'K={k}', 
                fontsize=7, ha='center', alpha=0.6,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    
    ax2.set_xlabel('Top-K Kunden', fontsize=10, fontweight='bold')
    ax2.set_ylabel('Anzahl überlappender Kunden', fontsize=10, fontweight='bold')
    ax2.set_title('Ranking Overlap vs. Baseline\n(Fläche = Impact-Stärke)', 
                  fontsize=11, fontweight='bold', pad=10)
    ax2.legend(fontsize=8, loc='upper left', framealpha=0.9)
    ax2.grid(alpha=0.25, linestyle=':', linewidth=0.8)
    ax2.set_xlim(0, 200)
    ax2.set_ylim(0, 200)
    
    # --- PLOT 3 (rechts oben): 3-Metriken-Vergleich ---
    ax3 = fig.add_subplot(gs[0, 3])
    
    # Daten sammeln
    metrics_data = []
    for weight_name in weight_vectors.keys():
        weight_metrics = stats_df[stats_df['Metrik'].str.contains(weight_name)]
        if len(weight_metrics) > 0:
            metrics_data.append({
                'Gewichtung': weight_name,
                'Range': weight_metrics['range'].mean(),
                'Std': weight_metrics['std'].mean(),
                'Share_Neg': weight_metrics['share_negative'].mean() * 100
            })
    
    metrics_df = pd.DataFrame(metrics_data)
    
    # Normalisierung für Radar-ähnliche Darstellung
    metrics_df['Range_norm'] = (metrics_df['Range'] - metrics_df['Range'].min()) / \
                                (metrics_df['Range'].max() - metrics_df['Range'].min())
    metrics_df['Std_norm'] = (metrics_df['Std'] - metrics_df['Std'].min()) / \
                              (metrics_df['Std'].max() - metrics_df['Std'].min())
    metrics_df['Neg_norm'] = metrics_df['Share_Neg'] / 100
    
    # Gestapeltes Balkendiagramm
    x_pos = np.arange(len(metrics_df))
    width = 0.6
    
    # Drei Metriken übereinander
    p1 = ax3.barh(x_pos, metrics_df['Range_norm'], width, 
                  label='Range (norm)', color='#2ECC71', alpha=0.8)
    p2 = ax3.barh(x_pos, metrics_df['Std_norm'], width, 
                  left=metrics_df['Range_norm'],
                  label='Std (norm)', color='#3498DB', alpha=0.8)
    p3 = ax3.barh(x_pos, metrics_df['Neg_norm'], width,
                  left=metrics_df['Range_norm'] + metrics_df['Std_norm'],
                  label='% Neg', color='#E74C3C', alpha=0.8)
    
    ax3.set_yticks(x_pos)
    ax3.set_yticklabels(metrics_df['Gewichtung'], fontsize=8)
    ax3.set_xlabel('Diskriminierungs-Score\n(normalisiert, summiert)', fontsize=9)
    ax3.set_title('Multi-Metrik\nDiskriminierung', fontsize=11, fontweight='bold', pad=10)
    ax3.legend(fontsize=7, loc='lower right')
    ax3.grid(axis='x', alpha=0.3, linestyle='--')
    
    # Werte als Text
    for i, row in metrics_df.iterrows():
        total = row['Range_norm'] + row['Std_norm'] + row['Neg_norm']
        ax3.text(total + 0.05, i, f'{total:.2f}', 
                va='center', fontsize=8, fontweight='bold')
    
    # --- PLOT 4 (unten): Impact Summary Tabelle ---
    ax4 = fig.add_subplot(gs[1, :])
    ax4.axis('tight')
    ax4.axis('off')
    
    if impact_summary is not None:
        # Top 3 Impact-Gewichtungen
        top_impact = impact_summary.nlargest(3, 'Impact_Score')
        
        table_data = []
        for _, row in top_impact.iterrows():
            table_data.append([
                row['Gewichtung'],
                f"{row['Impact_Score']:.3f}",
                f"{row['Durchschn_Abweichung']:.1f}",
                f"K={int(row['Erste_signifikante_Abweichung_K']) if pd.notna(row['Erste_signifikante_Abweichung_K']) else 'N/A'}"
            ])
        
        table = ax4.table(
            cellText=table_data,
            colLabels=['Top Impact Gewichtungen', 'Impact Score', 'Ø Abweichung', '1. Sign. Abw.'],
            cellLoc='center',
            loc='center',
            colWidths=[0.3, 0.15, 0.15, 0.15]
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 1.8)
        
        # Header-Styling
        for i in range(4):
            cell = table[(0, i)]
            cell.set_facecolor('#34495E')
            cell.set_text_props(weight='bold', color='white')
        
        # Erste Zeile (Rank 1) hervorheben
        for i in range(4):
            cell = table[(1, i)]
            cell.set_facecolor('#F39C12')
            cell.set_text_props(weight='bold')
    else:
        # Fallback: Legende/Interpretation
        interpretation_text = (
            "INTERPRETATION:\n"
            "• Separation Power: Höher = Bessere Trennung ähnlich/unähnlich\n"
            "• Overlap: Größere Fläche = Stärkerer Einfluss der Gewichtung\n"
            "• Diskriminierung: Höher = Mehr Differenzierungspotenzial"
        )
        ax4.text(0.5, 0.5, interpretation_text,
                ha='center', va='center', fontsize=9,
                bbox=dict(boxstyle='round,pad=1', facecolor='lightgray', alpha=0.3))
    
    # Gesamt-Titel
    plt.suptitle(
        'Experiment 2: Averaging vs. Weighting - Kompaktübersicht',
        fontsize=15, fontweight='bold', y=0.98
    )
    
    return fig


# Alternative: Noch kompaktere 2x2 Version
def visualize_exp2_summary_compact(
    separation_matrix: pd.DataFrame,
    overlap_curves_result: dict,
    stats_df: pd.DataFrame,
    weight_vectors: dict,
    figsize=(14, 10)
):
    """
    Noch kompaktere 2x2 Version mit maximaler Informationsdichte.
    """
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    ax1, ax2, ax3, ax4 = axes.flatten()
    
    # --- 1. Separation + Ranking (kombiniert) ---
    heatmap_data = separation_matrix.set_index('Gewichtung').copy()
    if 'Durchschnitt' in heatmap_data.columns:
        heatmap_data = heatmap_data.drop(columns=['Durchschnitt'])
    
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        center=heatmap_data.values.mean(),
        cbar_kws={'label': 'Separation'},
        linewidths=0.5,
        ax=ax1,
        vmin=0
    )
    ax1.set_title('Separation Power Matrix', fontsize=12, fontweight='bold')
    
    # --- 2. Overlap-Kurven ---
    curves = overlap_curves_result['curves']
    optimal = overlap_curves_result['optimal_curve']
    x = list(range(1, len(optimal) + 1))
    
    ax2.plot(x, optimal, 'k--', linewidth=2, label='Optimal', alpha=0.7)
    
    colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6']
    for i, (weight_name, curve) in enumerate(curves.items()):
        ax2.plot(x, curve, linewidth=2, label=weight_name,
                color=colors[i % len(colors)], alpha=0.8)
        # Fläche
        ax2.fill_between(x, curve, optimal, 
                        color=colors[i % len(colors)], alpha=0.08)
    
    ax2.set_xlabel('Top-K', fontsize=10)
    ax2.set_ylabel('Overlap', fontsize=10)
    ax2.set_title('Ranking Overlap (Fläche = Impact)', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=8, loc='upper left')
    ax2.grid(alpha=0.3)
    ax2.set_xlim(0, 200)
    
    # --- 3. Verteilungs-Boxplots ---
    # Sammle Werte für Boxplots
    boxplot_data = []
    labels = []
    
    for weight_name in weight_vectors.keys():
        weight_metrics = stats_df[stats_df['Metrik'].str.contains(weight_name)]
        if len(weight_metrics) > 0:
            boxplot_data.append(weight_metrics['range'].values)
            labels.append(weight_name)
    
    bp = ax3.boxplot(boxplot_data, labels=labels, patch_artist=True,
                     showmeans=True, meanline=True)
    
    # Farben
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    ax3.set_ylabel('Range (Spannweite)', fontsize=10)
    ax3.set_title('Spannweiten-Verteilung', fontsize=12, fontweight='bold')
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(axis='y', alpha=0.3)
    
    # --- 4. Ranking-Tabelle ---
    ax4.axis('off')
    
    # Erstelle Ranking nach Durchschnitt
    sep_matrix_with_avg = separation_matrix.copy()
    if 'Durchschnitt' not in sep_matrix_with_avg.columns:
        metric_cols = [c for c in sep_matrix_with_avg.columns if c != 'Gewichtung']
        sep_matrix_with_avg['Durchschnitt'] = sep_matrix_with_avg[metric_cols].mean(axis=1)
    
    ranked = sep_matrix_with_avg.sort_values('Durchschnitt', ascending=False)
    
    table_data = []
    for i, (_, row) in enumerate(ranked.head(5).iterrows(), 1):
        # Sammle Info
        weight_name = row['Gewichtung']
        avg_sep = row['Durchschnitt']
        
        # Range aus stats_df
        weight_metrics = stats_df[stats_df['Metrik'].str.contains(weight_name)]
        avg_range = weight_metrics['range'].mean() if len(weight_metrics) > 0 else 0
        
        table_data.append([
            f"#{i}",
            weight_name,
            f"{avg_sep:.3f}",
            f"{avg_range:.3f}"
        ])
    
    table = ax4.table(
        cellText=table_data,
        colLabels=['Rank', 'Gewichtung', 'Separation', 'Range'],
        cellLoc='center',
        loc='center',
        colWidths=[0.1, 0.4, 0.25, 0.25]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.5)
    
    # Styling
    for i in range(4):
        table[(0, i)].set_facecolor('#34495E')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Gold für #1
    for i in range(4):
        table[(1, i)].set_facecolor('#FFD700')
        table[(1, i)].set_text_props(weight='bold')
    
    ax4.set_title('Top-5 Gewichtungen', fontsize=12, fontweight='bold', pad=20)
    
    plt.suptitle(
        'Experiment 2: Gesamtübersicht (Kompakt)',
        fontsize=14, fontweight='bold', y=0.98
    )
    plt.tight_layout()
    
    return fig

In [ ]:

# fig1 = visualize_exp2_summary_enhanced(
#     separation_matrix=separation_matrix,
#     overlap_curves_result=overlap_curves,
#     stats_df=results["exp2_phase3_stats_model0"]['stats_df'],
#     weight_vectors=weight_vectors,
#     impact_summary=impact_summary  # Optional
# )

# Version 2: Compact (noch platzsparender)
for i in [0,1]:
    fig2 = visualize_exp2_summary_compact(
        separation_matrix=separation_matrix[i],
        overlap_curves_result=overlap_curves[i],
        stats_df=results[f"exp2_phase3_stats_model{i}"]['stats_df'],
        weight_vectors=weight_vectors
    )




In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
for i in [0,1]:
    df = profiles[i]["categorical_feature_profile"]
    
    plot_df = (
        df[~df["feature"].str.startswith("key")]
        .groupby(["feature", "candidate_value"], as_index=False)
        .agg(
            conditional_churn_rate=("conditional_churn_rate", "mean"),
            count=("overall_count", "mean"),
        )
    )
    
    # optional: nur häufige Ausprägungen
    plot_df = plot_df[plot_df["count"] > 200]
    
    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=plot_df,
        x="conditional_churn_rate",
        y="candidate_value",
        hue="feature",
        dodge=False
    )
    plt.xlabel("Conditional churn rate")
    plt.ylabel("Feature value")
    plt.title("Conditional churn rates by feature value")
    plt.tight_layout()
    plt.show()


In [ ]:
heat_df = df[
    (df["feature"].isin(["country", "products_number", "active_member"]))
]

heat_pivot = heat_df.pivot(
    index="key1",
    columns="feature",
    values="conditional_churn_rate"
)

plt.figure(figsize=(8, 5))
sns.heatmap(
    heat_pivot,
    annot=True,
    fmt=".2f",
    cmap="Reds",
    cbar_kws={"label": "Conditional churn rate"}
)
plt.title("Churn risk profile of top-10 candidates")
plt.ylabel("Customer")
plt.xlabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
scatter_df = (
    df[~df["feature"].str.startswith("key")]
    .drop_duplicates(subset=["feature", "candidate_value"])
)

plt.figure(figsize=(6, 6))
sns.scatterplot(
    data=scatter_df,
    x="overall_freq",
    y="churned_freq",
    hue="feature"
)

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("Overall frequency")
plt.ylabel("Frequency among churned customers")
plt.title("Overrepresentation of feature values among churners")
plt.tight_layout()
plt.show()
